In [1]:
import pandas as pd
import numpy as np
import pickle
from scipy.sparse import csr_matrix
from implicit import als
import os

data_path   = '../data'
models_path = '../models'

# Load light ratings
df = pd.read_csv(f'{data_path}/clean_ratings_light.csv')
print(f"Loaded: {df.shape}")

# Build correct matrix (users x products)
df['confidence'] = df['rating'].apply(lambda x: 1 + 40 * x)

train_users    = df['user_id'].unique()
train_products = df['product_id'].unique()

train_user_encoder    = {uid: idx for idx, uid in enumerate(train_users)}
train_user_decoder    = {idx: uid for uid, idx in train_user_encoder.items()}
train_product_encoder = {pid: idx for idx, pid in enumerate(train_products)}
train_product_decoder = {idx: pid for pid, idx in train_product_encoder.items()}

user_idx    = df['user_id'].map(train_user_encoder)
product_idx = df['product_id'].map(train_product_encoder)

valid       = user_idx.notna() & product_idx.notna()
df_clean    = df[valid].copy()
user_idx    = user_idx[valid].astype(int)
product_idx = product_idx[valid].astype(int)

train_matrix = csr_matrix(
    (df_clean['confidence'], (user_idx, product_idx)),
    shape=(len(train_user_encoder), len(train_product_encoder))
)

print(f"Matrix shape: {train_matrix.shape}")

# Train ALS
print("Training ALS...")
train_als = als.AlternatingLeastSquares(
    factors=50, regularization=0.1,
    iterations=30, use_gpu=False
)
train_als.fit(train_matrix)
print(f"✅ ALS trained: {train_als.user_factors.shape}")

Loaded: (72696, 4)
Matrix shape: (10327, 2717)
Training ALS...


c:\Users\bvais\anaconda3\envs\ml\lib\site-packages\implicit\cpu\als.py:95: RuntimeWarning: Intel MKL BLAS is configured to use 4 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'MKL_NUM_THREADS=1' or by callng 'threadpoolctl.threadpool_limits(1, "blas")'. Having MKL use a threadpool can lead to severe performance issues
  check_blas_config()


  0%|          | 0/30 [00:00<?, ?it/s]

✅ ALS trained: (10327, 50)


In [2]:
deploy_path = '../models/deploy'
os.makedirs(deploy_path, exist_ok=True)

# Save retrained ALS
with open(f'{deploy_path}/train_als.pkl', 'wb') as f:
    pickle.dump(train_als, f)

# Save train matrix
with open(f'{deploy_path}/train_matrix.pkl', 'wb') as f:
    pickle.dump(train_matrix, f)

# Save encoders
with open(f'{deploy_path}/train_encoders.pkl', 'wb') as f:
    pickle.dump({
        'train_user_encoder'    : train_user_encoder,
        'train_user_decoder'    : train_user_decoder,
        'train_product_encoder' : train_product_encoder,
        'train_product_decoder' : train_product_decoder,
    }, f)

# Copy existing models needed
import shutil
for fname in ['combined_matrix.pkl',
              'product_profiles.pkl',
              'product_idx_map.pkl',
              'meta_lookup.pkl']:
    shutil.copy(f'{models_path}/{fname}',
                f'{deploy_path}/{fname}')
    print(f"  Copied: {fname}")

# Save light CSV to deploy folder
df[['user_id','product_id','rating']].to_csv(
    f'{deploy_path}/ratings.csv', index=False
)

print("\n✅ All deploy models saved!")
print("\nFiles in deploy folder:")
for f in os.listdir(deploy_path):
    size = os.path.getsize(f'{deploy_path}/{f}') / 1024**2
    print(f"  {f:35} → {size:.1f} MB")

  Copied: combined_matrix.pkl
  Copied: product_profiles.pkl
  Copied: product_idx_map.pkl
  Copied: meta_lookup.pkl

✅ All deploy models saved!

Files in deploy folder:
  combined_matrix.pkl                 → 9.7 MB
  meta_lookup.pkl                     → 1.0 MB
  product_idx_map.pkl                 → 0.1 MB
  product_profiles.pkl                → 50.4 MB
  ratings.csv                         → 2.1 MB
  train_als.pkl                       → 2.5 MB
  train_encoders.pkl                  → 0.3 MB
  train_matrix.pkl                    → 0.8 MB


In [3]:
total = sum(
    os.path.getsize(f'{deploy_path}/{f}')
    for f in os.listdir(deploy_path)
) / 1024**2

print(f"Total deploy size: {total:.1f} MB")
print("\nTarget: under 400MB for Render free tier")

if total > 400:
    print("⚠️ Too large — we'll need to upload to Google Drive")
else:
    print("✅ Small enough to include in repo directly")

Total deploy size: 66.9 MB

Target: under 400MB for Render free tier
✅ Small enough to include in repo directly


In [5]:
import pickle
import os

# Full path
path = r"D:\VCUBE NOTES\projects\Amazon(RS)\amazon-recsy\models\deploy\product_profiles.pkl"

# Load
with open(path, 'rb') as f:
    product_profiles = pickle.load(f)

print(f"Type    : {type(product_profiles)}")
print(f"Shape   : {product_profiles.shape}")
print(f"Columns : {product_profiles.columns.tolist()}")

# Keep only essential columns
essential_cols = ['product_id', 'avg_rating', 'review_count']
product_profiles_slim = product_profiles[essential_cols].copy()

# Save slim version back
with open(path, 'wb') as f:
    pickle.dump(product_profiles_slim, f)

size = os.path.getsize(path) / 1024**2
print(f"\n✅ Slim size: {size:.1f} MB")

Type    : <class 'pandas.core.frame.DataFrame'>
Shape   : (2717, 6)
Columns : ['product_id', 'all_reviews', 'all_summaries', 'avg_rating', 'review_count', 'content']

✅ Slim size: 0.1 MB
